In [8]:
# Load CSV
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/raw/training_data.csv")

In [10]:
print("Loaded dataset:", df.shape)
print(f"{df.shape[0]} patients, {df.shape[1]} features")

Loaded dataset: (942, 23)
942 patients, 23 features


In [11]:
# Check for dataset imbalance
df["Outcome"].value_counts(normalize=True)

Outcome
Normal      0.515924
Abnormal    0.484076
Name: proportion, dtype: float64

In [13]:
# Check split distribution
# (if you already merged splits, skip this step)
splits = pd.read_csv("../data/processed/patient_splits.csv")
df = df.merge(splits, on="Patient ID", how="left")

# --------------------------------------------------
# Compute counts
# --------------------------------------------------
# counts = (
#     df.groupby(["split", "Outcome"])
#     .size()
#     .reset_index(name="count")
# )

pivot = df.pivot_table(
    index="split",
    columns="Outcome",
    values="Patient ID",
    aggfunc="count",
    fill_value=0
)

# convert to percentages
pivot_percent = pivot.div(pivot.sum(axis=1), axis=0) * 100

print(pivot_percent)

Outcome   Abnormal     Normal
split                        
test     49.295775  50.704225
train    48.406677  51.593323
val      47.517730  52.482270


# Analyze feature_dataset

In [48]:
df = pd.read_csv("../data/processed/feature_dataset.csv")

In [49]:
print("Loaded dataset:", df.shape)
print(f"{df.shape[0]} patients, {df.shape[1]} features")

Loaded dataset: (3163, 109)
3163 patients, 109 features


In [50]:
# One hot encoding for age and recording location
df = pd.get_dummies(df, columns=["Age"], drop_first=True)

df = pd.get_dummies(df, columns=["recording_location"], drop_first=True)

df = pd.get_dummies(df, columns=["Murmur"], drop_first=True)

df["Sex"] = df["Sex"].map({
    "Female": 0,
    "Male": 1
})


drop_cols = ["Patient ID", "Outcome", "split", "file", "Campaign", "Additional ID", "Height", "Weight"]
df = df.drop(columns=drop_cols)

In [51]:
print("Loaded dataset:", df.shape)
print(f"{df.shape[0]} patients, {df.shape[1]} features")

Loaded dataset: (3163, 107)
3163 patients, 107 features


In [53]:
print("Total of NaN values:", df.isnull().values.sum())

print ("Total of rows with NaN values:", df.isnull().any(axis=1).sum())

null_rows = df[df.isnull().any(axis=1)]
print(null_rows.index.tolist())

Total of NaN values: 0
Total of rows with NaN values: 0
[]


In [54]:
print(df.head)

<bound method NDFrame.head of       Sex  Pregnancy status       rms  peak  variance          mean       std  \
0       1             False  0.051440   1.0  0.002646 -1.738083e-11  0.051440   
1       1             False  0.035360   1.0  0.001250  8.283024e-11  0.035360   
2       1             False  0.044045   1.0  0.001940  2.475065e-10  0.044045   
3       1             False  0.048863   1.0  0.002388  7.233574e-11  0.048863   
4       1             False  0.038561   1.0  0.001487  1.804437e-10  0.038561   
...   ...               ...       ...   ...       ...           ...       ...   
3158    0             False  0.214775   1.0  0.046128 -2.082622e-11  0.214775   
3159    1             False  0.045776   1.0  0.002095 -1.189198e-10  0.045776   
3160    1             False  0.040254   1.0  0.001620  2.628914e-10  0.040254   
3161    1             False  0.033521   1.0  0.001124  3.130496e-11  0.033521   
3162    1             False  0.037949   1.0  0.001440 -2.349011e-10  0.037949  